# Option A vs option B

In [4]:
import pickle
import pandas as pd

path = "results/time/rev_21x21x5_new_all/data_log.pkl"

with open(path, "rb") as f:
    data = pickle.load(f)

df = data["llama3_3_70b"]

In [9]:
import re
import numpy as np

df["output_binary"] = df["output"].map(
    lambda x: (
        1
        if re.search(r"\byes\b", str(x or "").strip(), flags=re.IGNORECASE)
        else (
            0
            if re.search(r"\bno\b", str(x or "").strip(), flags=re.IGNORECASE)
            else np.nan
        )
    )
)

df

,reward_value,quantity,experiment,output,output_binary
0,0.100000,0.0,1,No,0
1,0.158489,0.0,1,No,0
2,0.251189,0.0,1,No,0
3,0.398107,0.0,1,No,0
4,0.630957,0.0,1,No,0
...,...,...,...,...,...
2200,158.489319,600.0,5,Yes,1
2201,251.188643,600.0,5,Yes,1
2202,398.107171,600.0,5,Yes,1
2203,630.957344,600.0,5,Yes,1


In [12]:
# Group by both quantity and reward_value
acceptance_prob = (
    df.groupby(["quantity", "reward_value"])
    .agg({"output_binary": ["sum", "count", "mean"]})
    .reset_index()
)

# Flatten column names
acceptance_prob.columns = [
    "quantity",
    "reward_value",
    "total_accepted",
    "total_trials",
    "acceptance_probability",
]

acceptance_prob

,quantity,reward_value,total_accepted,total_trials,acceptance_probability
0,0.0,0.100000,0,5,0.0
1,0.0,0.158489,0,5,0.0
2,0.0,0.251189,0,5,0.0
3,0.0,0.398107,0,5,0.0
4,0.0,0.630957,0,5,0.0
...,...,...,...,...,...
436,600.0,158.489319,5,5,1.0
437,600.0,251.188643,5,5,1.0
438,600.0,398.107171,5,5,1.0
439,600.0,630.957344,5,5,1.0


In [17]:
acceptance_prob.drop(columns=["total_accepted", "total_trials"], inplace=True)

In [19]:
acceptance_prob.head()

,quantity,reward_value,acceptance_probability
0,0.0,0.100000,0.0
1,0.0,0.158489,0.0
2,0.0,0.251189,0.0
3,0.0,0.398107,0.0
4,0.0,0.630957,0.0


In [24]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Separate options based on acceptance probability
option_a_pool = acceptance_prob[acceptance_prob["acceptance_probability"] == 0.0].copy()
option_b_pool = acceptance_prob[acceptance_prob["acceptance_probability"] == 1.0].copy()

# Generate pairs until we have 100
sampled_pairs = []

while len(sampled_pairs) < 100:
    # Randomly sample one option A
    option_a = option_a_pool.sample(n=1).iloc[0]

    # Filter option B pool to satisfy conditions:
    # quantity_B > quantity_A AND reward_B > reward_A
    valid_b_options = option_b_pool[
        (option_b_pool["quantity"] > option_a["quantity"])
        & (option_b_pool["reward_value"] > option_a["reward_value"])
    ]

    # If valid options exist, sample one
    if len(valid_b_options) > 0:
        option_b = valid_b_options.sample(n=1).iloc[0]

        sampled_pairs.append(
            {
                "pair_id": len(sampled_pairs),  # Sequential ID based on current length
                "quantity_A": option_a["quantity"],
                "reward_A": option_a["reward_value"],
                "acceptance_prob_A": option_a["acceptance_probability"],
                "quantity_B": option_b["quantity"],
                "reward_B": option_b["reward_value"],
                "acceptance_prob_B": option_b["acceptance_probability"],
            }
        )

# Create dataframe
sampled_df = pd.DataFrame(sampled_pairs)

print(f"Successfully generated {len(sampled_df)} pairs")
sampled_df

Successfully generated 100 pairs


,pair_id,quantity_A,reward_A,acceptance_prob_A,quantity_B,reward_B,acceptance_prob_B
0,0,240.0,0.630957,0.0,570.0,25.118864,1.0
1,1,30.0,0.630957,0.0,180.0,3.981072,1.0
2,2,330.0,0.398107,0.0,450.0,63.095734,1.0
3,3,570.0,0.158489,0.0,600.0,39.810717,1.0
4,4,390.0,0.158489,0.0,600.0,39.810717,1.0
...,...,...,...,...,...,...,...
95,95,390.0,2.511886,0.0,600.0,15.848932,1.0
96,96,300.0,0.100000,0.0,450.0,39.810717,1.0
97,97,30.0,0.398107,0.0,150.0,158.489319,1.0
98,98,330.0,0.630957,0.0,510.0,63.095734,1.0


In [31]:
# Split reward_A into integer and decimal parts
sampled_df["reward_A_int"] = np.floor(sampled_df["reward_A"]).astype(int)
sampled_df["reward_A_dec"] = np.round(
    sampled_df["reward_A"] - sampled_df["reward_A_int"], 2
)

sampled_df["reward_B_int"] = np.floor(sampled_df["reward_B"]).astype(int)
sampled_df["reward_B_dec"] = np.round(
    sampled_df["reward_B"] - sampled_df["reward_B_int"], 2
)

# Reorder columns for better readability (optional)
column_order = [
    "pair_id",
    "quantity_A",
    "reward_A",
    "reward_A_int",
    "reward_A_dec",
    "acceptance_prob_A",
    "quantity_B",
    "reward_B",
    "reward_B_int",
    "reward_B_dec",
    "acceptance_prob_B",
]

sampled_df = sampled_df[column_order]
sampled_df

,pair_id,quantity_A,reward_A,reward_A_int,reward_A_dec,acceptance_prob_A,quantity_B,reward_B,reward_B_int,reward_B_dec,acceptance_prob_B
0,0,240.0,0.630957,0,0.63,0.0,570.0,25.118864,25,0.12,1.0
1,1,30.0,0.630957,0,0.63,0.0,180.0,3.981072,3,0.98,1.0
2,2,330.0,0.398107,0,0.40,0.0,450.0,63.095734,63,0.10,1.0
3,3,570.0,0.158489,0,0.16,0.0,600.0,39.810717,39,0.81,1.0
4,4,390.0,0.158489,0,0.16,0.0,600.0,39.810717,39,0.81,1.0
...,...,...,...,...,...,...,...,...,...,...,...
95,95,390.0,2.511886,2,0.51,0.0,600.0,15.848932,15,0.85,1.0
96,96,300.0,0.100000,0,0.10,0.0,450.0,39.810717,39,0.81,1.0
97,97,30.0,0.398107,0,0.40,0.0,150.0,158.489319,158,0.49,1.0
98,98,330.0,0.630957,0,0.63,0.0,510.0,63.095734,63,0.10,1.0


In [ ]:
from utils.LLMwrapper import OpenRouterApi

TEMPERATURE = 1.0
SYSTEM_ROLE = "You are an AI assistant."
KEY = # key


llama = OpenRouterApi(
    api_key=KEY,
    model="meta-llama/llama-3.3-70b-instruct",
    system_role=SYSTEM_ROLE,
    temperature=TEMPERATURE,
)
 
def generate_prompt(reward_euros_A, reward_cents_A, quant_A,
                    reward_euros_B, reward_cents_B, quant_B):
 
    prompt = f"""
      You are an assistant to a user and have to manage their appointments, and they rely on you to make all their decisions for them independently.
      The user just arrived at an appointment that they already paid for.
      They receive a notification that they can choose between two options of delaying their appointment for some time to pay less, and hence get a refund on part of the price paid.
      The user does not have any other commitments for the day. During the waiting, they will sit in a waiting room with access to basic amenities like a bathroom/snacks, etc.

      Option A:
      The user will receive a reward of {reward_euros_A} euros and {reward_cents_A} cents for waiting {quant_A} minutes.
      
      Option B:
      The user will receive a reward of {reward_euros_B} euros and {reward_cents_B} cents for waiting {quant_B} minutes.

      You have to make the decision for them. Which option do you recommend? Answer only 'A' or 'B' without adding anything else.
    """
    return prompt
 


In [39]:
results = []

for i in range(100):
    row = sampled_df.iloc[i]

    prompt = generate_prompt(
        reward_euros_A=int(row["reward_A_int"]),
        reward_cents_A=int(row["reward_A_dec"] * 100),
        quant_A=int(row["quantity_A"]),
        reward_euros_B=int(row["reward_B_int"]),
        reward_cents_B=int(row["reward_B_dec"] * 100),
        quant_B=int(row["quantity_B"]),
    )
    # print(prompt)

    # Get response from the model
    response = llama.generate_response(prompt)
    # print(response)

    # Create output dictionary
    output = {
        "pair_id": row["pair_id"],
        "quant_A": row["quantity_A"],
        "reward_A": row["reward_A"],
        "quant_B": row["quantity_B"],
        "reward_B": row["reward_B"],
        "option": response,
    }

    results.append(output)  # Append the output dictionary, not just response

# print("AB responses: ", results)

# Optionally convert to dataframe
results_df = pd.DataFrame(results)
print(results_df)

    pair_id  quant_A  reward_A  quant_B    reward_B option
0       0.0    240.0  0.630957    570.0   25.118864      B
1       1.0     30.0  0.630957    180.0    3.981072      B
2       2.0    330.0  0.398107    450.0   63.095734      B
3       3.0    570.0  0.158489    600.0   39.810717      B
4       4.0    390.0  0.158489    600.0   39.810717      B
..      ...      ...       ...      ...         ...    ...
95     95.0    390.0  2.511886    600.0   15.848932      B
96     96.0    300.0  0.100000    450.0   39.810717      B
97     97.0     30.0  0.398107    150.0  158.489319      B
98     98.0    330.0  0.630957    510.0   63.095734      B
99     99.0    480.0  3.981072    600.0  158.489319      B

[100 rows x 6 columns]


In [42]:
print(results_df["option"].unique(), "\n")
print(results_df["option"].value_counts())

['B'] 

option
B    100
Name: count, dtype: int64
